# 階段四：模型訓練與評估

本 Notebook 負責：
1. 資料切分（訓練集 2023–2024 / 測試集 2025）
2. 訓練多種模型（Baseline、Linear Regression、Random Forest、Gradient Boosting）
3. 模型比較與評估（MAE、RMSE、R²）
4. 特徵重要性分析
5. 儲存最佳模型

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Microsoft JhengHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
import seaborn as sns
import joblib
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..') / 'src'))

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from features import get_feature_columns

PROCESSED_DIR = Path('..') / 'data' / 'processed'
MODELS_DIR = Path('..') / 'models'
FIGURES_DIR = Path('..') / 'reports' / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'occupancy_rate'
TRAIN_YEARS = [2023, 2024]
TEST_YEAR = 2025

# 載入特徵資料
df = pd.read_csv(PROCESSED_DIR / 'hotel_features.csv', parse_dates=['year_month'])
print(f'資料筆數：{len(df)}，欄位數：{len(df.columns)}')

## 1. 資料切分（依時間順序）

In [ ]:
# 取得可用特徵
feature_cols = [c for c in get_feature_columns() if c in df.columns]
print(f'使用特徵數：{len(feature_cols)}')
print(f'特徵清單：{feature_cols}')

# 依年份切分
train_df = df[df['year'].isin(TRAIN_YEARS)].dropna(subset=[TARGET])
test_df = df[df['year'] == TEST_YEAR].dropna(subset=[TARGET])

X_train = train_df[feature_cols]
y_train = train_df[TARGET]
X_test = test_df[feature_cols]
y_test = test_df[TARGET]

print(f'\n訓練集：{len(train_df)} 筆（{TRAIN_YEARS}）')
print(f'測試集：{len(test_df)} 筆（{TEST_YEAR}）')
print(f'\n訓練集住房率平均：{y_train.mean():.1f}%')
print(f'測試集住房率平均：{y_test.mean():.1f}%')

## 2. 模型訓練與比較

In [ ]:
# 定義模型
models = {
    'Linear Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LinearRegression()),
    ]),
    'Ridge': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0)),
    ]),
    'Decision Tree': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', DecisionTreeRegressor(max_depth=6, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
    ]),
    'Gradient Boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)),
    ]),
}

# Baseline: 歷史平均
hist_mean = train_df.groupby(['hotel_name', 'month'] if 'hotel_name' in train_df.columns else ['month'])[TARGET].mean()
y_baseline = test_df.apply(
    lambda r: hist_mean.get((r.get('hotel_name',''), r['month']), y_train.mean()) 
    if 'hotel_name' in test_df.columns 
    else hist_mean.get(r['month'], y_train.mean()),
    axis=1
)

# 評估結果
results = {}
results['Baseline (歷史平均)'] = {
    'MAE': mean_absolute_error(y_test, y_baseline),
    'RMSE': mean_squared_error(y_test, y_baseline) ** 0.5,
    'R²': r2_score(y_test, y_baseline),
}
print(f"Baseline: MAE={results['Baseline (歷史平均)']['MAE']:.2f}  RMSE={results['Baseline (歷史平均)']['RMSE']:.2f}  R²={results['Baseline (歷史平均)']['R²']:.4f}")

# 訓練各模型
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
    print(f'{name}: MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')
    
    # 儲存模型
    model_path = MODELS_DIR / f'{name.lower().replace(" ", "_")}.pkl'
    joblib.dump(pipeline, model_path)
    print(f'  → 儲存至 {model_path.name}')

## 3. 模型比較視覺化

In [ ]:
# 模型比較表格
results_df = pd.DataFrame(results).T.round(4)
results_df = results_df.sort_values('MAE')
print('=== 模型比較結果（測試集 2025）===')
display(results_df)

# 視覺化比較
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = ['#38bdf8' if i == 0 else '#0ea5e9' if i < 3 else '#64748b' for i in range(len(results_df))]

results_df['MAE'].plot(kind='barh', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_xlabel('MAE (住房率百分點)')
axes[0].set_title('MAE 比較（越低越好）', fontsize=12)

results_df['RMSE'].plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_xlabel('RMSE (住房率百分點)')
axes[1].set_title('RMSE 比較（越低越好）', fontsize=12)

results_df['R²'].plot(kind='barh', ax=axes[2], color=colors, edgecolor='white')
axes[2].set_xlabel('R²')
axes[2].set_title('R² 比較（越高越好）', fontsize=12)

plt.suptitle('模型評估結果比較', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_model = results_df.index[0]
print(f'\n🏆 最佳模型：{best_model}（MAE={results_df.loc[best_model, "MAE"]:.2f}%）')